# SOB4ES - Extractor automatico de resultados entre ramas

Este notebook automatiza lo que se ha estado haciendo a mano hasta ahora: leer los notebooks de una rama de git (sin necesidad de hacer `checkout`, usando `git show rama:archivo`), extraer las metricas de evaluacion, el numero de variables activas, y detectar automaticamente dos problemas que ya han aparecido varias veces de forma manual:

1. **Variables que no coinciden** entre lo que dice `FEATURES_AUTORIZADAS` y lo que realmente se cargo en `X_train` (columna comentada pero no re-ejecutado).
2. **Celdas ejecutadas fuera de orden** (`execution_count` no creciente de arriba a abajo), que es la causa raiz de los resultados "pegados" de una ejecucion anterior (el bug que se detecto en `regressorchain.ipynb` y `mlp_custom_loss.ipynb`).

Compara dos ramas cualesquiera (p. ej. `main` contra `prueba1-iteracion-2`) para los 8 notebooks de modelos, y genera la misma tabla comparativa que se ha ido pidiendo manualmente en el chat.

**Requisito:** tener el repositorio clonado localmente (funciona sobre tu copia local de git, sin necesidad de subir nada a ningun sitio). No hace falta hacer `git checkout` de las ramas - se leen los archivos directamente del historial de git.


## 0. Configuracion

In [60]:
import os
import pandas as pd
from IPython.display import display

# --- EDITA ESTO ---
GIT_REPO_PATH = "."  # Usar '.' si ejecutas el notebook dentro del repositorio

# Define las ramas que quieras comparar (puedes poner 2, 3, 5 o las que necesites)
BRANCHES = [
    "model-prep-var",
    "model-prep-var-1",
    "model-prep-var-2",
    "model-prep-var-1-2",
    "model-prep-var-3",
    "model-prep-var-1-3",
    # "model-prep-var-x-y",   Añade más ramas según necesites
]

NOTEBOOKS = [
    "xgboost_model.ipynb",
    "xgb_multisalida.ipynb",
    "rf_model.ipynb",
    "rf_multisalida.ipynb",
    "regressorchain.ipynb",
    "reg-model.ipynb",
    "mlp_multisalida.ipynb",
    "mlp_custom_loss.ipynb",
]

NOTEBOOKS_SUBDIR = ""
TARGETS_PRIORITARIOS = ["earthworm_shannon_z", "earthworm_richness_z"]

## 1. Funciones de lectura desde git

`obtener_notebook_desde_git` usa `git show <rama>:<archivo>` para leer el contenido de un archivo tal y como esta en una rama concreta, sin tocar el working directory ni hacer checkout. Si el repo es remoto y no lo tienes clonado, hay una alternativa comentada al final de la celda usando la API de GitHub (`raw.githubusercontent.com`).

In [61]:
def obtener_notebook_desde_git(repo_path, branch, filename, subdir=""):
    """Lee un .ipynb tal y como está en una rama concreta, vía `git show`, sin checkout."""
    ruta_relativa = os.path.join(subdir, filename) if subdir else filename
    try:
        resultado = subprocess.run(
            ["git", "-C", repo_path, "show", f"{branch}:{ruta_relativa}"],
            capture_output=True,
            text=True,
            check=True,
        )
    except subprocess.CalledProcessError as e:
        print(
            f"  [ERROR] No se pudo leer {filename} en la rama {branch}: {e.stderr.strip()}"
        )
        return None
    return json.loads(resultado.stdout)


def listar_ramas(repo_path):
    """Útil para comprobar el nombre exacto de las ramas disponibles (locales y remotas)."""
    resultado = subprocess.run(
        ["git", "-C", repo_path, "branch", "-a"],
        capture_output=True,
        text=True,
    )
    print(resultado.stdout)

## 2. Funciones de extraccion

- `extraer_features_activas`: parsea la lista `FEATURES_AUTORIZADAS` y separa las lineas comentadas (excluidas) de las activas.
- `extraer_shape_xtrain`: busca el print de `X_train: (filas, columnas)` para saber cuantas variables se usaron realmente en el entrenamiento.
- `detectar_celdas_desordenadas`: compara el `execution_count` de las celdas de codigo en el orden en que aparecen en el notebook; si no es creciente, señala un posible problema de ejecucion (resultados de una corrida anterior, no de la actual).
- `extraer_metricas_targets`: busca en los outputs de texto, **a partir del marcador "Evaluacion final sobre eval.csv"**, las lineas con R2/RMSE/MAE para los targets pedidos. Restringir la busqueda a ese bloque es importante: sin eso, el regex puede confundirse con el R2 de entrenamiento/CV impreso en las celdas de tuning (mucho mas alto por sobreajuste) y dar una lectura falsa. Soporta dos formatos: modelos single-target (R2+RMSE+MAE por target) y modelos multisalida (solo R2 por target, con RMSE/MAE unicamente a nivel global).

In [62]:
def extraer_features_activas(nb_json):
    for c in nb_json["cells"]:
        src = "".join(c.get("source", []))
        if "FEATURES_AUTORIZADAS =" in src:
            m = re.search(r"FEATURES_AUTORIZADAS\s*=\s*\[(.*?)\]", src, re.S)
            if not m:
                continue
            lineas = [l.strip() for l in m.group(1).split("\n") if l.strip()]
            activas, excluidas = [], []
            for linea in lineas:
                nombre_m = re.search(r"'([^']+)'", linea)
                if not nombre_m:
                    continue
                nombre = nombre_m.group(1)
                if linea.startswith("#"):
                    excluidas.append(nombre)
                else:
                    activas.append(nombre)
            return activas, excluidas
    return [], []


def extraer_shape_xtrain(nb_json):
    for c in nb_json["cells"]:
        for out in c.get("outputs", []):
            texto = "".join(out.get("text", []))
            m = re.search(r"X_train:\s*\((\d+),\s*(\d+)\)", texto)
            if m:
                return int(m.group(1)), int(m.group(2))
    return None, None


def detectar_celdas_desordenadas(nb_json):
    """Devuelve una lista de avisos si el execution_count no es creciente."""
    avisos = []
    ultimo_exec = None
    for i, c in enumerate(nb_json["cells"]):
        if c.get("cell_type") != "code":
            continue
        exec_count = c.get("execution_count")
        if exec_count is None:
            continue
        if ultimo_exec is not None and exec_count < ultimo_exec:
            avisos.append(
                f"celda #{i} tiene execution_count={exec_count}, "
                f"menor que una celda anterior ({ultimo_exec}) -> posible resultado obsoleto"
            )
        ultimo_exec = exec_count
    return avisos


MARCADOR_EVAL = "Evaluacion final sobre eval.csv"


def extraer_metricas_targets(nb_json, targets):
    """Extrae R2/RMSE/MAE de la evaluación final para cada target."""
    resultados = {}
    patron_completo = {
        t: re.compile(
            rf"{re.escape(t)}\s+([\-0-9.]+)\s+([\-0-9.]+)\s+([\-0-9.]+)"
        )
        for t in targets
    }
    patron_solo_r2 = {
        t: re.compile(rf"{re.escape(t)}\s+([\-0-9.]+)\s*$", re.M)
        for t in targets
    }
    patron_global = re.compile(
        r"R2\s+global:\s*([\-0-9.]+).*?RMSE\s+global:\s*([\-0-9.]+).*?MAE\s+global:\s*([\-0-9.]+)",
        re.S,
    )

    for c in nb_json["cells"]:
        for out in c.get("outputs", []):
            texto_completo = "".join(out.get("text", []))
            if MARCADOR_EVAL not in texto_completo:
                continue
            texto = texto_completo[texto_completo.index(MARCADOR_EVAL) :]

            g = patron_global.search(texto)
            global_metrics = (
                {
                    "r2_global": float(g.group(1)),
                    "rmse_global": float(g.group(2)),
                    "mae_global": float(g.group(3)),
                }
                if g
                else None
            )

            for t in targets:
                if t in resultados:
                    continue
                m = patron_completo[t].search(texto)
                if m:
                    resultados[t] = {
                        "r2": float(m.group(1)),
                        "rmse": float(m.group(2)),
                        "mae": float(m.group(3)),
                    }
                    continue
                m2 = patron_solo_r2[t].search(texto)
                if m2:
                    resultados[t] = {
                        "r2": float(m2.group(1)),
                        "rmse": None,
                        "mae": None,
                        "global": global_metrics,
                    }
    return resultados

## 3. Extraccion por notebook y por rama

In [63]:
def analizar_notebook(repo_path, branch, filename, subdir, targets):
    nb_json = obtener_notebook_desde_git(repo_path, branch, filename, subdir)
    if nb_json is None:
        return None

    activas, excluidas = extraer_features_activas(nb_json)
    filas, columnas = extraer_shape_xtrain(nb_json)
    metricas = extraer_metricas_targets(nb_json, targets)
    avisos_orden = detectar_celdas_desordenadas(nb_json)

    return {
        "n_features_lista": len(activas),
        "features_excluidas": excluidas,
        "x_train_shape": (filas, columnas),
        "coherente": (
            (columnas == len(activas)) if columnas is not None else None
        ),
        "metricas": metricas,
        "avisos_orden": avisos_orden,
    }


resultados_por_rama = {}

for branch in BRANCHES:
    print(f"Leyendo notebooks de la rama: {branch}...")
    resultados_por_rama[branch] = {}
    for nb_name in NOTEBOOKS:
        print(f"  {nb_name}")
        resultados_por_rama[branch][nb_name] = analizar_notebook(
            GIT_REPO_PATH,
            branch,
            nb_name,
            NOTEBOOKS_SUBDIR,
            TARGETS_PRIORITARIOS,
        )
    print()

Leyendo notebooks de la rama: model-prep-var...
  xgboost_model.ipynb
  xgb_multisalida.ipynb
  rf_model.ipynb
  rf_multisalida.ipynb
  regressorchain.ipynb
  reg-model.ipynb
  mlp_multisalida.ipynb
  mlp_custom_loss.ipynb

Leyendo notebooks de la rama: model-prep-var-1...
  xgboost_model.ipynb
  xgb_multisalida.ipynb
  rf_model.ipynb
  rf_multisalida.ipynb
  regressorchain.ipynb
  reg-model.ipynb
  mlp_multisalida.ipynb
  mlp_custom_loss.ipynb

Leyendo notebooks de la rama: model-prep-var-2...
  xgboost_model.ipynb
  xgb_multisalida.ipynb
  rf_model.ipynb
  rf_multisalida.ipynb
  regressorchain.ipynb
  reg-model.ipynb
  mlp_multisalida.ipynb
  mlp_custom_loss.ipynb

Leyendo notebooks de la rama: model-prep-var-1-2...
  xgboost_model.ipynb
  xgb_multisalida.ipynb
  rf_model.ipynb
  rf_multisalida.ipynb
  regressorchain.ipynb
  reg-model.ipynb
  mlp_multisalida.ipynb
  mlp_custom_loss.ipynb

Leyendo notebooks de la rama: model-prep-var-3...
  xgboost_model.ipynb
  xgb_multisalida.ipynb


## 4. Validaciones automaticas

Antes de comparar metricas, se comprueba automaticamente lo que hasta ahora se ha ido revisando a mano:
- ¿El numero de columnas de `X_train` coincide con el numero de variables activas en `FEATURES_AUTORIZADAS`?
- ¿Hay celdas con `execution_count` fuera de orden (posible resultado obsoleto)?
- ¿Los valores de los targets prioritarios son sospechosamente identicos entre ambas ramas (posible notebook no re-ejecutado)?

In [64]:
print("=" * 100)
print("VALIDACIONES AUTOMÁTICAS")
print("=" * 100)

base_branch = BRANCHES[0]

for nb_name in NOTEBOOKS:
    print(f"\n[{nb_name}]")

    # 1. Comprobar coherencia y orden por cada rama
    for branch in BRANCHES:
        r = resultados_por_rama.get(branch, {}).get(nb_name)
        if r is None:
            print(f"  [ERROR - {branch}] No se pudo leer el notebook.")
            continue

        filas, columnas = r["x_train_shape"]
        if r["coherente"] is False:
            print(
                f"  [AVISO - {branch}] X_train tiene {columnas} columnas pero "
                f"FEATURES_AUTORIZADAS tiene {r['n_features_lista']} activas -> revisar notebook"
            )
        if r["avisos_orden"]:
            print(f"  [AVISO - {branch}] Celdas ejecutadas fuera de orden:")
            for a in r["avisos_orden"]:
                print(f"      - {a}")

    # 2. Comprobar si hay métricas idénticas respecto a la primera rama (baseline)
    r_base = resultados_por_rama.get(base_branch, {}).get(nb_name)
    if r_base:
        for comp_branch in BRANCHES[1:]:
            r_comp = resultados_por_rama.get(comp_branch, {}).get(nb_name)
            if not r_comp:
                continue
            for t in TARGETS_PRIORITARIOS:
                m_base = r_base["metricas"].get(t)
                m_comp = r_comp["metricas"].get(t)
                if (
                    m_base
                    and m_comp
                    and m_base.get("r2") == m_comp.get("r2")
                ):
                    print(
                        f"  [AVISO] {t}: R2 IDENTICO en '{base_branch}' y '{comp_branch}' ({m_base['r2']}) "
                        f"-> revisar si se re-ejecutó de verdad"
                    )

VALIDACIONES AUTOMÁTICAS

[xgboost_model.ipynb]

[xgb_multisalida.ipynb]

[rf_model.ipynb]

[rf_multisalida.ipynb]
  [AVISO - model-prep-var] Celdas ejecutadas fuera de orden:
      - celda #8 tiene execution_count=17, menor que una celda anterior (21) -> posible resultado obsoleto

[regressorchain.ipynb]

[reg-model.ipynb]
  [AVISO - model-prep-var] Celdas ejecutadas fuera de orden:
      - celda #8 tiene execution_count=5, menor que una celda anterior (7) -> posible resultado obsoleto

[mlp_multisalida.ipynb]

[mlp_custom_loss.ipynb]


## 5. Tabla comparativa final

In [65]:
filas_tabla = []
base_branch = BRANCHES[0]

for nb_name in NOTEBOOKS:
    fila = {"Notebook": nb_name}

    for t in TARGETS_PRIORITARIOS:
        # Obtener R2 de la rama base
        r_base = resultados_por_rama.get(base_branch, {}).get(nb_name)
        m_base = r_base["metricas"].get(t) if r_base else None
        r2_base = m_base["r2"] if m_base else None

        fila[f"{t}_R2 ({base_branch})"] = r2_base

        # R2 y Delta para las ramas a comparar
        for branch in BRANCHES[1:]:
            r_branch = resultados_por_rama.get(branch, {}).get(nb_name)
            m_branch = r_branch["metricas"].get(t) if r_branch else None
            r2_val = m_branch["r2"] if m_branch else None

            fila[f"{t}_R2 ({branch})"] = r2_val

            delta = (
                round(r2_val - r2_base, 4)
                if (r2_val is not None and r2_base is not None)
                else None
            )
            fila[f"{t}_delta ({branch})"] = delta

    filas_tabla.append(fila)

df_comparativa = pd.DataFrame(filas_tabla)

print(f"Comparación entre {len(BRANCHES)} ramas (Base: {base_branch}):\n")
print(df_comparativa.to_string(index=False))

# Guardar resultado en CSV
os.makedirs("output/comparativas", exist_ok=True)
nombre_salida = f"comparativa_{'_vs_'.join(BRANCHES)}.csv".replace("/", "-")
ruta_csv = f"output/comparativas/{nombre_salida}"
df_comparativa.to_csv(ruta_csv, index=False)
print(f"\nGuardado en: {ruta_csv}")

Comparación entre 6 ramas (Base: model-prep-var):

             Notebook  earthworm_shannon_z_R2 (model-prep-var)  earthworm_shannon_z_R2 (model-prep-var-1)  earthworm_shannon_z_delta (model-prep-var-1)  earthworm_shannon_z_R2 (model-prep-var-2)  earthworm_shannon_z_delta (model-prep-var-2)  earthworm_shannon_z_R2 (model-prep-var-1-2)  earthworm_shannon_z_delta (model-prep-var-1-2)  earthworm_shannon_z_R2 (model-prep-var-3)  earthworm_shannon_z_delta (model-prep-var-3)  earthworm_shannon_z_R2 (model-prep-var-1-3)  earthworm_shannon_z_delta (model-prep-var-1-3)  earthworm_richness_z_R2 (model-prep-var)  earthworm_richness_z_R2 (model-prep-var-1)  earthworm_richness_z_delta (model-prep-var-1)  earthworm_richness_z_R2 (model-prep-var-2)  earthworm_richness_z_delta (model-prep-var-2)  earthworm_richness_z_R2 (model-prep-var-1-2)  earthworm_richness_z_delta (model-prep-var-1-2)  earthworm_richness_z_R2 (model-prep-var-3)  earthworm_richness_z_delta (model-prep-var-3)  earthworm_richness_z_R

### 5.1.- Tabla de compraración general

In [70]:
filas_general = []
base_branch = BRANCHES[0]

for nb_name in NOTEBOOKS:
    fila = {"Notebook": nb_name}

    for branch in BRANCHES:
        res = resultados_por_rama.get(branch, {}).get(nb_name)

        if res:
            # Dimensiones de X_train
            shape_str = (
                f"{res['x_train_shape'][0]}x{res['x_train_shape'][1]}"
                if res["x_train_shape"][0]
                else "N/A"
            )
            fila[f"N_Vars ({branch})"] = res["n_features_lista"]
            fila[f"Shape ({branch})"] = shape_str

            # Extracción de métricas globales
            m_dict = res.get("metricas", {})
            r2_glob, rmse_glob, mae_glob = None, None, None

            # 1. Intentar obtener 'global' si el notebook es multisalida
            for t_info in m_dict.values():
                if t_info and t_info.get("global"):
                    r2_glob = t_info["global"].get("r2_global")
                    rmse_glob = t_info["global"].get("rmse_global")
                    mae_glob = t_info["global"].get("mae_global")
                    break

            # 2. Si es single-target, promediar el R2 de los targets evaluados
            if r2_glob is None and m_dict:
                r2_vals = [
                    v["r2"]
                    for v in m_dict.values()
                    if v and v.get("r2") is not None
                ]
                if r2_vals:
                    r2_glob = round(sum(r2_vals) / len(r2_vals), 4)

            fila[f"R2_Global ({branch})"] = r2_glob
            if rmse_glob is not None:
                fila[f"RMSE_Global ({branch})"] = rmse_glob
            if mae_glob is not None:
                fila[f"MAE_Global ({branch})"] = mae_glob

            # Calcular Delta R2 Global respecto al baseline
            if branch != base_branch:
                r2_base = fila.get(f"R2_Global ({base_branch})")
                fila[f"Delta_R2_Global ({branch})"] = (
                    round(r2_glob - r2_base, 4)
                    if (r2_glob is not None and r2_base is not None)
                    else None
                )
        else:
            fila[f"N_Vars ({branch})"] = "Error"
            fila[f"R2_Global ({branch})"] = None

    filas_general.append(fila)

df_general = pd.DataFrame(filas_general)

print("=" * 100)
print(f"1. TABLA COMPARATIVA GENERAL (Base: {base_branch})")
print("=" * 100)
print(df_general.to_markdown(index=False))

# Guardar CSV
os.makedirs("output/comparativas", exist_ok=True)
ruta_csv_gen = f"output/comparativas/general_{'_vs_'.join(BRANCHES)}.csv".replace(
    "/", "-"
)
df_general.to_csv(ruta_csv_gen, index=False)
print(f"\nGuardado en: {ruta_csv_gen}")

1. TABLA COMPARATIVA GENERAL (Base: model-prep-var)
| Notebook              |   N_Vars (model-prep-var) | Shape (model-prep-var)   |   R2_Global (model-prep-var) |   N_Vars (model-prep-var-1) | Shape (model-prep-var-1)   |   R2_Global (model-prep-var-1) |   Delta_R2_Global (model-prep-var-1) |   N_Vars (model-prep-var-2) | Shape (model-prep-var-2)   |   R2_Global (model-prep-var-2) |   Delta_R2_Global (model-prep-var-2) |   N_Vars (model-prep-var-1-2) | Shape (model-prep-var-1-2)   |   R2_Global (model-prep-var-1-2) |   Delta_R2_Global (model-prep-var-1-2) |   N_Vars (model-prep-var-3) | Shape (model-prep-var-3)   |   R2_Global (model-prep-var-3) |   Delta_R2_Global (model-prep-var-3) |   N_Vars (model-prep-var-1-3) | Shape (model-prep-var-1-3)   |   R2_Global (model-prep-var-1-3) |   Delta_R2_Global (model-prep-var-1-3) |   RMSE_Global (model-prep-var) |   MAE_Global (model-prep-var) |   RMSE_Global (model-prep-var-1) |   MAE_Global (model-prep-var-1) |   RMSE_Global (model-prep-var-2

### 5.2.- Comparación por targets

In [67]:
filas_targets = []
base_branch = BRANCHES[0]

for nb_name in NOTEBOOKS:
    for target in TARGETS_PRIORITARIOS:
        fila = {"Notebook": nb_name, "Target": target}

        # Métricas de referencia (Baseline)
        res_base = resultados_por_rama.get(base_branch, {}).get(nb_name)
        m_base = (
            res_base["metricas"].get(target)
            if (res_base and res_base.get("metricas"))
            else None
        )
        r2_base = m_base["r2"] if m_base else None

        for branch in BRANCHES:
            res = resultados_por_rama.get(branch, {}).get(nb_name)
            m = (
                res["metricas"].get(target)
                if (res and res.get("metricas"))
                else None
            )

            r2_val = m["r2"] if m else None
            rmse_val = m["rmse"] if m else None
            mae_val = m["mae"] if m else None

            fila[f"R2 ({branch})"] = r2_val

            if rmse_val is not None:
                fila[f"RMSE ({branch})"] = rmse_val
            if mae_val is not None:
                fila[f"MAE ({branch})"] = mae_val

            # Delta R2 respecto a la rama base
            if branch != base_branch:
                delta = (
                    round(r2_val - r2_base, 4)
                    if (r2_val is not None and r2_base is not None)
                    else None
                )
                fila[f"Delta_R2 ({branch})"] = delta

        filas_targets.append(fila)

df_targets = pd.DataFrame(filas_targets)

print("=" * 100)
print(f"2. TABLA COMPARATIVA POR TARGETS (Base: {base_branch})")
print("=" * 100)
print(df_targets.to_string(index=False))

# Guardar CSV
ruta_csv_targets = (
    f"output/comparativas/por_targets_{'_vs_'.join(BRANCHES)}.csv".replace(
        "/", "-"
    )
)
df_targets.to_csv(ruta_csv_targets, index=False)
print(f"\nGuardado en: {ruta_csv_targets}")

2. TABLA COMPARATIVA POR TARGETS (Base: model-prep-var)
             Notebook               Target  R2 (model-prep-var)  RMSE (model-prep-var)  MAE (model-prep-var)  R2 (model-prep-var-1)  RMSE (model-prep-var-1)  MAE (model-prep-var-1)  Delta_R2 (model-prep-var-1)  R2 (model-prep-var-2)  RMSE (model-prep-var-2)  MAE (model-prep-var-2)  Delta_R2 (model-prep-var-2)  R2 (model-prep-var-1-2)  RMSE (model-prep-var-1-2)  MAE (model-prep-var-1-2)  Delta_R2 (model-prep-var-1-2)  R2 (model-prep-var-3)  RMSE (model-prep-var-3)  MAE (model-prep-var-3)  Delta_R2 (model-prep-var-3)  R2 (model-prep-var-1-3)  RMSE (model-prep-var-1-3)  MAE (model-prep-var-1-3)  Delta_R2 (model-prep-var-1-3)
  xgboost_model.ipynb  earthworm_shannon_z               0.4946                 0.7096                0.5336                 0.4931                   0.7107                  0.5333                      -0.0015                 0.4906                   0.7124                  0.5351                      -0.0040    

## 6. Variables excluidas por rama

Muestra que variables estan comentadas (excluidas) en `FEATURES_AUTORIZADAS` en cada rama, para confirmar rapidamente que la rama de comparacion excluye la variable esperada (y solo esa) en los 8 notebooks.

In [68]:
for nb_name in NOTEBOOKS:
    print(f"\n--- {nb_name} ---")
    for branch in BRANCHES:
        res = resultados_por_rama.get(branch, {}).get(nb_name)
        if res:
            excluidas = res["features_excluidas"]
            print(f"  [{branch:25}] Excluidas ({len(excluidas)}): {excluidas}")
        else:
            print(f"  [{branch:25}] Error al leer notebook")


--- xgboost_model.ipynb ---
  [model-prep-var           ] Excluidas (0): []
  [model-prep-var-1         ] Excluidas (1): ['cu_z']
  [model-prep-var-2         ] Excluidas (1): ['ni_z']
  [model-prep-var-1-2       ] Excluidas (2): ['cu_z', 'ni_z']
  [model-prep-var-3         ] Excluidas (1): ['mo_z']
  [model-prep-var-1-3       ] Excluidas (2): ['cu_z', 'mo_z']

--- xgb_multisalida.ipynb ---
  [model-prep-var           ] Excluidas (0): []
  [model-prep-var-1         ] Excluidas (1): ['cu_z']
  [model-prep-var-2         ] Excluidas (1): ['ni_z']
  [model-prep-var-1-2       ] Excluidas (2): ['cu_z', 'ni_z']
  [model-prep-var-3         ] Excluidas (1): ['mo_z']
  [model-prep-var-1-3       ] Excluidas (2): ['cu_z', 'mo_z']

--- rf_model.ipynb ---
  [model-prep-var           ] Excluidas (0): []
  [model-prep-var-1         ] Excluidas (1): ['cu_z']
  [model-prep-var-2         ] Excluidas (1): ['ni_z']
  [model-prep-var-1-2       ] Excluidas (2): ['cu_z', 'ni_z']
  [model-prep-var-3         ] 